In [9]:
from datetime import date, datetime
from openpyxl import load_workbook
from openpyxl.pivot.fields import DateTimeField
from openpyxl.utils.datetime import to_excel


In [ ]:
ANALYSIS_DATE_CACHE_FIELD_INDEX = 12  # AnalysisDate in this workbook's pivot cache


def _calendar_day(value):
    if isinstance(value, datetime):
        return value.date()
    if isinstance(value, date):
        return value
    raise TypeError(f"Expected date or datetime, got {type(value)!r}")


def analysis_dates_in_pivot_cache(cache, field_idx=ANALYSIS_DATE_CACHE_FIELD_INDEX):
    """Map calendar date -> pivot cache shared-items index for AnalysisDate."""
    cf = cache.cacheFields[field_idx]
    out = {}
    for i, f in enumerate(cf.sharedItems._fields):
        if isinstance(f, DateTimeField) and f.v is not None:
            out[_calendar_day(f.v)] = i
    return out


def set_dashboard_analysis_date_filter(wb, target_date):
    """Set the Dashboard pivot's AnalysisDate report filter to ``target_date``.

    Updates (1) pivot field item hidden flags (``h`` on each ``<item>``) and
    (2) ``Dashboard!B1`` so the filter label matches. ``pageField@item`` is
    left unset so Excel (especially with multi-select page fields) does not
    show a repair dialog.

    ``target_date`` must already exist in the pivot cache (dates that appear in
    the source range). After saving, open in Excel and **refresh the pivot**
    (right-click → Refresh) so the table body totals match the filter.
    """
    day = _calendar_day(target_date)
    ws = wb["Dashboard"]
    p = ws._pivots[0]
    by_day = analysis_dates_in_pivot_cache(p.cache)
    if day not in by_day:
        keys = sorted(by_day.keys())
        raise KeyError(f"{day} not in pivot cache (have {len(keys)} dates, e.g. {keys[:3]} … {keys[-3:]})")
    x_sel = by_day[day]
    pf = p.pivotFields[ANALYSIS_DATE_CACHE_FIELD_INDEX]
    item_idx = None
    for i, item in enumerate(pf.items):
        if item.t != "data" or item.x is None:
            continue
        if item.x == x_sel:
            item.h = None
            item_idx = i
        else:
            item.h = True
    if item_idx is None:
        raise RuntimeError(f"No pivot item row for cache index {x_sel}")
    # Do not set pageFields[].item: with multipleItemSelectionAllowed on the page
    # field, Excel often stores the filter using item @h only. Adding/changing
    # pageField@item can make the part inconsistent and trigger "Repair PivotTable".
    if p.pageFields:
        p.pageFields[0].item = None
    ex = to_excel(day)
    ws["B1"] = int(ex) if ex == int(ex) else ex
    return {"date": day, "cache_index": x_sel, "page_item_index": item_idx}


wb = load_workbook("CADENT_SurveyTracker.xlsx")
set_dashboard_analysis_date_filter(wb, date(2026, 1, 10))  # change to any cached date
wb.save("CADENT_SurveyTracker.xlsx")  # or another path to avoid overwriting
